In [34]:
from forex_python.converter import CurrencyRates

def calculate_lot_size_auto(
    balance,
    risk_percent,
    entry_price,
    stop_loss_price,
    pair,
    account_currency="USD"
):
    """
    Universal lot size calculator for any forex pair.
    Automatically fetches live exchange rates via forex-python.

    balance:          account balance in account currency
    risk_percent:     risk per trade (e.g. 2 for 2%)
    entry_price:      trade entry price
    stop_loss_price:  stop loss price
    pair:             e.g. "USD/CAD", "EUR/JPY", "GBP/ZAR"
    account_currency: your account currency (default = USD)
    """

    c = CurrencyRates()
    base, quote = pair.split("/")

    # 1️⃣ Pip size (0.01 for JPY pairs, otherwise 0.0001)
    pip_size = 0.01 if "JPY" in quote else 0.0001

    # 2️⃣ Pip value in the quote currency (before conversion)
    if quote == account_currency:
        pip_value = pip_size * 100000
    elif base == account_currency:
        pip_value = (pip_size / entry_price) * 100000
    else:
        # Pip value in quote currency
        pip_value_quote = (pip_size / entry_price) * 100000
        # Convert to account currency
        conversion_rate = c.get_rate(quote, account_currency)
        pip_value = pip_value_quote * conversion_rate

    # 3️⃣ Risk amount
    risk_amount = balance * (risk_percent / 100)

    # 4️⃣ Stop loss distance in pips
    pip_distance = abs(entry_price - stop_loss_price) / pip_size

    # 5️⃣ Lot size
    lot_size = risk_amount / (pip_distance * pip_value)
    
    def calculate_required_margin(lot_size, pair, entry_price, leverage):
        """
        Calculates the margin required to open a position.
        Assumes 1 lot = 100,000 units of base currency.
        """
        base, quote = pair.split("/")
        position_value = lot_size * 100000 * entry_price  # value in quote currency
        margin_required = position_value / leverage
        return margin_required

    margin_required = calculate_required_margin(lot_size, pair, entry_price, 200)
    
    return lot_size, margin_required


In [35]:
balance = 100
risk_percent = 5
entry_price = 1.15244
stop_loss_price = 1.15200
pair = "USD/CAD"
pair = "EUR/USD"

lot_size, margin_required = calculate_lot_size_auto(balance, risk_percent, entry_price, stop_loss_price, pair)
print(f"Lot size: {lot_size:.3f} and margin required: {margin_required:.2f} USD")

Lot size: 0.114 and margin required: 65.48 USD
